In [0]:
#01.Deduplicate Customer IDs with a Set — Given a list of customer IDs extracted from a CSV file, return the unique IDs and count how many duplicates were removed.
#helper function
def deduplicate_cust_ids(id_list:list)->tuple:
    #extract length of list of ids
    records_in_list=len(id_list)

    #count and remove duplicates
    unique_records=set(id_list)
    number_of_duplicates=records_in_list-len(unique_records)

    #return dictionary that has list_id & duplicate id count
    return (unique_records,number_of_duplicates)
#main function
if __name__=='__main__':
    record_list=[
        "C-001", "C_002", "C-001", "C/003"
    ]

    result=deduplicate_cust_ids(record_list) 
    print(result)

({'C/003', 'C-001', 'C_002'}, 1)


In [0]:
#01.Deduplicate Customer IDs with a Set — Given a list of customer IDs extracted from a CSV file, return the unique IDs and count how many duplicates were removed.
import logging
logger=logging.getLogger(__name__)

def deduplicate_customer_ids(customer_ids:list[str])->tuple[set[str], int]:
    if not isinstance(customer_ids, list):
        raise TypeError("customer_ids must be list of non empty strings")
    for row_number, customer_id in enumerate(customer_ids, start=1):
        if not isinstance (customer_id, str):
            raise TypeError(
                f"customer id at row {row_number} must be a sting"
                f"received {type(customer_id).__name__}")
        if not customer_id.strip():
            raise ValueError(
                f"customer id at row {row_number} must not be blank or whitespace")
    unique_customer_ids=set(customer_ids)
    duplicate_count=len(customer_ids)-len(unique_customer_ids)

    logger.debug(
        "customer id deduplication complete: input_count =%d, unique_count=%d, "
        "duplicate_count=%d",
        len(customer_ids),
        len(unique_customer_ids),
        duplicate_count
    )
    return unique_customer_ids, duplicate_count

if __name__=="__main__":
        record_list= ["C-001", "C_002", "C-001", "C/003"]
        unique_ids, duplicates_removed = deduplicate_customer_ids(record_list)
        print(f"Unique IDs: {unique_ids}")
        print(f"Duplicates removed: {duplicates_removed}")

Unique IDs: {'C/003', 'C-001', 'C_002'}
Duplicates removed: 1


In [0]:
import pytest
#import deduplicate_customer_ids

def test_removes_duplicates_and_return_count()->None:
    customer_ids=["C-001", "C_002", "C-001", "C/003"]

    unique_ids, duplicate_count=deduplicate_customer_ids(customer_ids)
    assert unique_ids=={"C-001", "C_002", "C/003"}
    assert duplicate_count==1

def test_all_customer_ids_are_unique()->None:
    unique_ids, duplicate_count=deduplicate_customer_ids(["C-001", "C_002",  "C/003"])
    assert unique_ids=={"C001", "C002", "C003"}
    assert duplicate_count==3

def test_rejects_non_list_input()->None:
    with pytest.raises(TypeError, match="must be a list"):
        deduplicate_customer_ids("C001,C002")

@pytest.mark.parametrize(
    "invalid_customer_ids",
    [
        [None],
        ["C001", 123],
        ["C001", {"id": "C002"}],
    ],
)

def test_rejects_non_string_customer_ids(invalid_customer_ids: list[object]) -> None:
    with pytest.raises(TypeError, match="must be a string"):
        deduplicate_customer_ids(invalid_customer_ids)  # type: ignore[arg-type]



<module 'pytest' from '/databricks/python/lib/python3.12/site-packages/pytest/__init__.py'>

In [0]:
#2.Build a File Status Dictionary — Given a list of filenames and processing statuses, build a dictionary mapping each filename to its latest status.
import re

VALID_STATUS=['received', 'processed', 'failed', 'skipped']
VALID_EXTENSIONS=['csv', 'txt']

def file_processing_status(file_list:list)->dict:
    #validate if incoming object is list
    if not isinstance(file_list, list):
        raise TypeError(f"file_list must be of type list, received {type(file_list).__name__}")
    #validate if file name is non empty
    if len(file_list)==0:
        raise ValueError(f"has no files to process, empty list")
    #validate if status is valid
    output_dict={}
    for name, status in file_list:
        if status not in VALID_STATUS:
            raise ValueError(f"file had no valid status, expected statuses are {VALID_STATUS}")
         #validate if extension is valid
        extention=name.rsplit('.',1)[-1]
        if extention not in VALID_EXTENSIONS:
            raise ValueError(f"file name does not have valid extension, "
                             f"valid extensions are {VALID_EXTENSIONS}")
        output_dict[name]=status
    #return dictionary
    return output_dict

if __name__=="__main__":
    file_list=[("orders.csv", "processed"), ("customers.csv", "received"), ("orders.csv", "failed")]
    result=file_processing_status(file_list)
    print(result) 

{'orders.csv': 'failed', 'customers.csv': 'received'}


In [0]:
#2.Build a File Status Dictionary — Given a list of filenames and processing statuses, build a dictionary mapping each filename to its latest status.
import logging
from collections.abc import Sequence
logger=logging.getLogger(__name__)

VALID_STATUSES:frozenset[str]=frozenset(
    {"received", "processed", "failed", "skipped"}
)
VALID_EXTENSIONS:frozenset[str]=frozenset(
    {"csv", "txt"}
)
def build_file_status_dictionary(
    file_list:list[Sequence[object]]
)->dict[str,str]:
    """
    Return the latest processing status for each valid filename.

    The latest status is determined by input order: when a filename occurs
    more than once, the status in its final record is returned.

    Args:
        file_list: A non-empty list of two-item filename/status records.

    Raises:
        TypeError: If file_list, a filename, or a status has an invalid type.
        ValueError: If the list is empty or a record has invalid content.
    """
    if not isinstance(file_list, list):
        raise TypeError("file must be a list"
                        f"received type {type(file_list).__name__}")
    if not file_list:
        raise ValueError("file_list must contain at least one file-status record")
    latest_status_by_file:dict[str, str]={}

    for index, record in enumerate(file_list):
        if (
            not isinstance(record, Sequence)
            or isinstance(record, (str, bytes))
            or len(record)!=2
        ):
            raise ValueError(
                f"Record at index {index} must be a two-item filename/status record")
        filename, status=record

        if not isinstance(filename, str):
            raise TypeError(
                f"filename at index {index} must be a string"
                f"received {type(filename).__name__}")
        filename=filename.strip()
        if not filename:
            raise ValueError(f"Filename at index {index} cannot be blank")

        if "." not in filename or filename.endswith("."):
            raise ValueError(
                f"File name at index {index} must include extension {filename!r}")
        extension=filename.rsplit(".",1)[1].lower()
        if extension not in VALID_EXTENSIONS:
            raise ValueError(
                f"unsupported extension at index {index}: {extension!r}"
                f"allowed extensions : {sorted(VALID_EXTENSIONS)}")
        if not isinstance(status, str):
            raise TypeError(
                f"status at indes {index} must be a string"
                f"received {type(status).__name__}")
        normalized_status=status.strip().lower()
        if normalized_status not in VALID_STATUSES:
            raise ValueError(
                f"invalid status at index {index}: {status!r}"
                f"allowed statuses are : {sorted(VALID_STATUSES)}"
                )
        latest_status_by_file[filename]=normalized_status
    logger.info(
        "built file status dictionary: input_records=%d unique_files=%d",
        len(file_list),
        len(latest_status_by_file)
    )
    return latest_status_by_file

if __name__=='__main__':
    file_list=[("daily_sales_2026-10-23.csv", "received"), ("daily_sales_2026-10-23.csv", "reprocessed")]
    for file in file_list:
        result=build_file_status_dictionary(file_list)
    print(result)

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-7257770636680025>, line 82
     80 file_list=[("daily_sales_2026-10-23.csv", "received"), ("daily_sales_2026-10-23.csv", "reprocessed")]
     81 for file in file_list:
---> 82     result=build_file_status_dictionary(file_list)
     83 print(result)

File <command-7257770636680025>, line 67, in build_file_status_dictionary(file_list)
     65     normalized_status=status.strip().lower()
     66     if normalized_status not in VALID_STATUSES:
---> 67         raise ValueError(
     68             f"invalid status at index {index}: {status!r}"
     69             f"allowed statuses are : {sorted(VALID_STATUSES)}"
     70             )
     71     latest_status_by_file[filename]=normalized_status
     72 logger.info(
     73     "built file status dictionary: input_records=%d unique_files=%d",
     74     len(file_list),
     75

In [0]:
import pytest 
def test_builds_dicitonary_for_unique_valid_files()->None:
     records=[
         ("orders.csv", "processed"),
        ("customers.txt", "received"),
     ]
     result=build_file_status_dictionary(records)

     assert result=={
        "orders.csv": "processed",
        "customers.txt": "received",
     }

def test_last_status_wins_for_duplicate_filename()->None:
    records=[
        ("orders.csv","received"),
        ("orders.csv", "processing") if False else ("orders.csv", "failed"),
        ("orders.csv","processed")
    ]
    result=build_file_status_dictionary(records)

    assert result=={"orders.csv", "processed"}

def test_accepts_uppercase_extension_and_normalizes_status()->None:
    result=build_file_status_dictionary([("ORDERS.CSV","PROCESSED")])

    assert result=={"orders.csv", "processed"}

@pytest.mark.parametrize(
    ("invalid_input", "expected_exception"),
    [
        (None, TypeError),
        ({"orders.csv":"processed"}, TypeError),
        ("orders.csv", TypeError),
        ([], ValueError),
    ],
)
def test_rejects_invalid_input(
    invalid_input:object, 
    expected_exception:type[Exception],
)->None:
    with pytest.raises(expected_exception):
        build_file_status_dictionary(invalid_input)

@pytest.mark.parametrize(
    "malformed_record",
    [
        (),
        ("orders.csv",),
        ("orders.csv", "processed", "extra"),
        "orders.csv",
        None,
    ],
)

def test_reject_malformed_records(malformed_record:object)->None:
    with pytest.raises(ValueError, match="index 0"):
        build_file_status_dictionary([malformed_record])

@pytest.mark.parametrize(
    "invalid_filename",
    [
        "",
        " ",
        "orders",
        "orders.",
        "orders.json",
        433,
        None
    ],
)

def test_rejects_invelid_filename(
    invalid_filename:object)->None:
    expected_exception=TypeError if not isinstance(invalid_filename, str) else ValueError

    with pytest.raises(expected_exception):
        build_file_status_dictionary([(invalid_filename, "processed")])

@pytest.mark.parametrize(
    "invalid_statuses",
    [
        "",
        "running",
        "processing",
        43,
        None
    ],
)
def test_rejects_invalid_statuses(invalid_status:object)->None:
    expected_exception=TypeError if not isinstance(invalid_status, str) else ValueError

    with pytest.raises(expected_exception):
        build_file_status_dictionary([("orders.csv", invalid_status)])

def test_preserves_distinct_path_with_same_base_filename()->None:
    result=build_file_status_dictionary(
        [
            ("incoming/orders.csv", "received"),
            ("archive/orders.csv", "processed"),
        ]
    )
    assert result=={
        "incoming/orders.csv": "received",
        "archive/orders.csv": "processed",
    }


In [0]:
pytest

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5527599958673138>, line 1
----> 1 pytest

NameError: name 'pytest' is not defined

In [0]:
#3.Represent Fixed Schema Metadata with Tuples — Store column name, data type, and nullable flag for each source column using tuples and validate that every tuple has exactly three fields.
from collections.abc import Sequence
import logging
logger=logging.getLogger(__name__)
#logging.getLogger(__name__)

#Valid data type & flags
VALID_DATATYPES:frozenset[str]=frozenset({'INTEGER', 'VARCHAR', 'DATE', 'STRING', 'NUMBER', 'TIMESTAMP'})
VALID_NULL_FLAGS:frozenset[bool]=frozenset({True, False})

#helper function
def schema_metadata_validation(column_tuple:tuple[object, object, object])->tuple[str, str, bool]:
    ''' this function validate metadata. 
    Assumption:
         Only valid data types defined in global variable will be accepted 
    Args:
        column_tuple: A non-empty tuple of  of three elements including
        column name, data type, and nullable flag.
    Raises:
        TypeError: If file_list, a filename, or a status has an invalid type.
        ValueError: If the list is empty or a record has invalid content.
    Returns: 
        Tuple of valid column name, data type, and nullable flag.'''
    #type sequence
    if not isinstance(column_tuple, tuple):
        raise TypeError(f"Expected a tuple but received {type(column_tuple).__name__}")
    #non None or empty
    if not column_tuple or column_tuple==None:
        raise ValueError(f"inbound field cannot be None or empty") 
    #length of tuple
    tuple_len=len(column_tuple)
    if tuple_len!=3:
        raise ValueError(f"inbound field expected to have 3 elements received {tuple_len}")
    #unpack tuple
    column_name, data_type, flag=column_tuple
    #validate & normalize column name
    if not isinstance(column_name, str):
        raise TypeError(f"column_name expected to be string received {type(column_name).__name__}")
    normalize_column_name=column_name.strip()
    if not normalize_column_name:
        raise ValueError("Column name cannot be empty")
    normalize_column_name=normalize_column_name.lower()
    #validate data type
    if not isinstance(data_type, str):
        raise TypeError(f"data_type expected to be string received {type(data_type).__name__}")
    normalize_data_type=data_type.strip()
    if not normalize_data_type:
        raise ValueError("data type cannot be empty")
    normalize_data_type=normalize_data_type.upper()
    if normalize_data_type not in VALID_DATATYPES:
        raise ValueError(f"this is not a valid data type, expected data types {VALID_DATATYPES}")
    #validate flag
    if not isinstance(flag, bool):
        raise TypeError(f"flag expected to be string received {type(flag).__name__}")    
    if flag not in VALID_NULL_FLAGS:
        raise ValueError(f"this is not valid flag, expected flag type {VALID_NULL_FLAGS}")
    logger.info(
        "Validated tuple in metadata: column_name=%d data_type=%d nullable_flag=%d",
        normalize_column_name,
        normalize_data_type,
        flag
    )
    #return tuple 
    return (normalize_column_name,normalize_data_type,flag)

#main function
if __name__=='__main__':
    column_tuple= ("customer_id", "INTEGER", False, '')
    result=schema_metadata_validation(column_tuple)
    print(result)

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-8750777105073421>, line 69
     67 if __name__=='__main__':
     68     column_tuple= ("customer_id", "INTEGER", False, '')
---> 69     result=schema_metadata_validation(column_tuple)
     70     print(result)

File <command-8750777105073421>, line 33, in schema_metadata_validation(column_tuple)
     31 tuple_len=len(column_tuple)
     32 if tuple_len!=3:
---> 33     raise ValueError(f"inbound field expected to have 3 elements received {tuple_len}")
     34 #unpack tuple
     35 column_name, data_type, flag=column_tuple

ValueError: inbound field expected to have 3 elements received 4

In [0]:
#3.Represent Fixed Schema Metadata with Tuples — Store column name, data type, and nullable flag for each source column using tuples and validate that every tuple has exactly three fields.
#from __future__ import annotations

import logging
from collections.abc import Iterable

logger=logging.getLogger(__name__)

VALID_DATA_TYPES:frozenset[str]=frozenset(
    {"INTEGER", "VARCHAR", "DATE", "STRING", "NUMBER", "TIMESTAMP"})
ColumnMetadata=tuple[str, str, bool]

def validate_column_metadata(record:object)->ColumnMetadata:
    """Validate and normalize one source-column metadata tuple.

    Expected tuple format:
        (column_name, data_type, nullable)

    Args:
        record: A tuple containing exactly three values.

    Returns:
        A normalized tuple of (lowercase_column_name, uppercase_data_type, nullable).

    Raises:
        TypeError: If the record or a field has an invalid type.
        ValueError: If the record has an invalid length or invalid content.
    """
    if not isinstance(record, tuple):
        raise TypeError(f"metadata record must be tuple, received {type(record).__name__}")
    if len(record)!=3:
        raise ValueError(f"metadata tuple must contain exactly 3 fields received {len(record)}")
    raw_column_name, raw_data_type, nullable=record
    if not isinstance(raw_column_name, str):
        raise TypeError(f"column name must be string, received {type(raw_column_name).__name__}")
    column_name=raw_column_name.strip().lower()
    if not column_name:
        raise ValueError("column name cannot be blank")
    if not isinstance(raw_data_type, str):
        raise TypeError(f"data type name must be a string, received {type(raw_data_type).__name__}")
    data_type=raw_data_type.strip().upper()
    if not data_type:
        raise ValueError("data type cannot be blank")
    if data_type not in VALID_DATA_TYPES:
        valid_types = ", ".join(sorted(VALID_DATA_TYPES))
        raise ValueError(
            f"Unsupported data_type '{data_type}'. Allowed values: {valid_types}."
        )
    if not isinstance(nullable, bool):
        raise TypeError(f"nullable must be a bool received {type(nullable).__name__}")
    logger.debug(
        "validated schema metadata : column_name=%s data_type=%s nullable=%s",
        column_name,
        data_type, 
        nullable
    )
    return column_name, data_type, nullable

def validate_schema_metadata(records:Iterable[object])->tuple[ColumnMetadata,...]:
        validated_records: list[ColumnMetadata] = []
        seen_column_names: set[str] = set()

        for index, record in enumerate(records):
            try:
                validated_record = validate_column_metadata(record)
            except (TypeError, ValueError) as error:
                raise type(error)(f"Invalid metadata at index {index}: {error}") from error

            column_name = validated_record[0]
            if column_name in seen_column_names:
                raise ValueError(
                    f"Duplicate column_name after normalization at index {index}: "
                    f"'{column_name}'."
                )

            seen_column_names.add(column_name)
            validated_records.append(validated_record)

        return tuple(validated_records)


if __name__=="__main__":
    source_schema=[
        (" customer_id ", "INTEGER", False),
        ("customer_id", "varchar", True),
        ("created_at", "timestamp", False),
    ]
    print(validate_schema_metadata(source_schema)) 

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-8750777105073422>, line 88
     82 if __name__=="__main__":
     83     source_schema=[
     84         (" customer_id ", "INTEGER", False),
     85         ("customer_id", "varchar", True),
     86         ("created_at", "timestamp", False),
     87     ]
---> 88     print(validate_schema_metadata(source_schema)) 

File <command-8750777105073422>, line 71, in validate_schema_metadata(records)
     69 column_name = validated_record[0]
     70 if column_name in seen_column_names:
---> 71     raise ValueError(
     72         f"Duplicate column_name after normalization at index {index}: "
     73         f"'{column_name}'."
     74     )
     76 seen_column_names.add(column_name)
     77 validated_records.append(validated_record)

ValueError: Duplicate column_name after normalization at index 1: 'customer_id'.